In [0]:
from pyspark.sql.functions import *

In [0]:
import sys
import importlib
sys.path.append('/Workspace/Users/kiranmanam9490@outlook.com/customer360')

import common.configuration
importlib.reload(common.configuration)
common.configuration.dbutils = dbutils

from common.configuration import Configuration

conf = Configuration()

In [0]:
df = spark.read.format('parquet').load(conf.base_url + '/bronze/web_activities')


In [0]:
df = df.withColumn(
    "session_time",
    coalesce(
        try_to_date(col("session_time"), "yyyy/MM/dd"),
        try_to_date(col("session_time"), "dd-MM-yyyy"),
        try_to_date(col("session_time"), "yyyy-MM-dd"),
        try_to_date(col("session_time"), "yyyyMMdd"),
        try_to_date(col("session_time"), "MM-dd-yyyy"),
        try_to_date(col("session_time"), "dd/MM/yyyy"),
    )
)

In [0]:
df = df.withColumn('page_viewed',lower(col('page_viewed')))\
.withColumn('device_type',initcap(col('device_type')))


In [0]:
df =df.dropDuplicates(["session_id","customer_id"])
df = df.dropna(subset=["session_id","customer_id"])

In [0]:
df = df.withColumn("customer_id", col("customer_id").cast("int"))

In [0]:
df.write.format("delta").mode("overwrite").save(conf.base_url + '/silver/web_activities')